# Call Center Data Cleaning

Cleaning the raw call center dataset and loading it into SQLite for analysis in SQL and Power BI.

In [20]:
%pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\frank\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd

In [22]:
# Loaded the raw dataset
df = pd.read_csv('Call Center Data.csv')
df.head()

,Index,Incoming Calls,Answered Calls,Answer Rate,Abandoned Calls,Answer Speed (AVG),Talk Duration (AVG),Waiting Time (AVG),Service Level (20 Seconds)
0,1,217,204,94.01%,13,0:00:17,0:02:14,0:02:45,76.28%
1,2,200,182,91.00%,18,0:00:20,0:02:22,0:06:55,72.73%
2,3,216,198,91.67%,18,0:00:18,0:02:38,0:03:50,74.30%
3,4,155,145,93.55%,10,0:00:15,0:02:29,0:03:12,79.61%
4,5,37,37,100.00%,0,0:00:03,0:02:06,0:00:35,97.30%


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1251 entries, 0 to 1250
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Index                       1251 non-null   int64 
 1   Incoming Calls              1251 non-null   int64 
 2   Answered Calls              1251 non-null   int64 
 3   Answer Rate                 1251 non-null   object
 4   Abandoned Calls             1251 non-null   int64 
 5   Answer Speed (AVG)          1251 non-null   object
 6   Talk Duration (AVG)         1251 non-null   object
 7   Waiting Time (AVG)          1251 non-null   object
 8   Service Level (20 Seconds)  1251 non-null   object
dtypes: int64(4), object(5)
memory usage: 88.1+ KB


In [24]:
df.describe()

,Index,Incoming Calls,Answered Calls,Abandoned Calls
count,1251.000000,1251.000000,1251.000000,1251.000000
mean,626.000000,198.539568,176.845723,21.693845
std,361.276902,156.534195,115.612080,59.671955
min,1.000000,5.000000,5.000000,0.000000
25%,313.500000,123.000000,114.000000,3.000000
50%,626.000000,177.000000,166.000000,8.000000
75%,938.500000,233.000000,214.500000,16.000000
max,1251.000000,1575.000000,909.000000,704.000000


In [25]:
# Removed the '%' symbol and converted these columns from text to float
df['Answer Rate'] = df['Answer Rate'].str.replace('%', '').astype(float)
df['Service Level (20 Seconds)'] = df['Service Level (20 Seconds)'].str.replace('%', '').astype(float)

In [26]:
# Converted the time columns from text (0:02:14) to total seconds
df['Talk Duration (AVG)'] = pd.to_timedelta(df['Talk Duration (AVG)']).dt.total_seconds()
df['Answer Speed (AVG)'] = pd.to_timedelta(df['Answer Speed (AVG)']).dt.total_seconds()
df['Waiting Time (AVG)'] = pd.to_timedelta(df['Waiting Time (AVG)']).dt.total_seconds()

In [27]:
# Calculated the abandon rate as a percentage of incoming calls
df['Abandon Rate'] = df['Abandoned Calls'] / df['Incoming Calls'] * 100

In [28]:
# Created a flag to check if the SLA was met, using 80% as the target
df['SLA Met'] = df['Service Level (20 Seconds)'] >= 80
df['SLA Met'].mean()

np.float64(0.36610711430855314)

In [29]:
# Renamed the columns to snake_case to avoid issues when importing to other tools
df = df.rename(columns={
    'Incoming Calls': 'incoming_calls',
    'Answered Calls': 'answered_calls',
    'Answer Rate': 'answer_rate',
    'Abandoned Calls': 'abandoned_calls',
    'Answer Speed (AVG)': 'answer_speed_avg',
    'Talk Duration (AVG)': 'talk_duration_avg',
    'Waiting Time (AVG)': 'waiting_time_avg',
    'Service Level (20 Seconds)': 'service_level_20_seconds',
    'Abandon Rate': 'abandon_rate',
    'SLA Met': 'sla_met'
})

In [30]:
# Checked the cleaned dataset
df.head()

,Index,incoming_calls,answered_calls,answer_rate,abandoned_calls,answer_speed_avg,talk_duration_avg,waiting_time_avg,service_level_20_seconds,abandon_rate,sla_met
0,1,217,204,94.01,13,17.0,134.0,165.0,76.28,5.990783,False
1,2,200,182,91.00,18,20.0,142.0,415.0,72.73,9.000000,False
2,3,216,198,91.67,18,18.0,158.0,230.0,74.30,8.333333,False
3,4,155,145,93.55,10,15.0,149.0,192.0,79.61,6.451613,False
4,5,37,37,100.00,0,3.0,126.0,35.0,97.30,0.000000,True


In [31]:
# Created a connection to SQLite and wrote the DataFrame as a table
import sqlite3
conn = sqlite3.connect('callcenter.db')
df.to_sql('calls', conn, if_exists='replace', index=False)

1251

In [32]:
# Confirmed the data loaded correctly (should return 1251)
pd.read_sql('SELECT COUNT(*) FROM calls', conn)

,COUNT(*)
0,1251


In [33]:
conn.close()